# 16 — Readout Calibration

## Purpose

Perform the complete readout calibration pipeline: characterize the IQ state distributions, optimize discrimination thresholds, integration weights, readout amplitude/frequency, and CLEAR waveform parameters to maximize measurement fidelity and QND-ness.

## Methodology

1. **IQ Blob** — scatter-plot g/e/f state distributions in the IQ plane
2. **CLEAR readout registration** — register the CLEAR depopulation waveform
3. **Square readout baseline** — measure with standard square pulse for comparison
4. **CLEAR readout test** — measure with CLEAR pulse and compare to square
5. **Raw g/e trace** — time-resolved IQ readout for signal arrival verification
6. **g/e discrimination** — optimize the threshold for binary state assignment
7. **Integrated trace** — demodulation window optimization
8. **κ measurement** — extract readout resonator linewidth from ring-down
9. **Residual photon Ramsey** — measure residual photon population after readout
10. **Butterfly measurement** — QND characterization (back-action analysis)
11. **Amplitude & length optimization** — 2D sweep for max SNR
12. **Frequency optimization** — sweep readout frequency for optimal discrimination
13. **CLEAR variant comparison** — side-by-side CLEAR vs. square performance
14. **Weight optimization** — optimize integration weights for maximal fidelity
15. **Full calibration** — end-to-end calibration combining all readout parameters

## Expected Outcomes

- Clear separation of g/e/f clusters in the IQ plane
- Discrimination fidelity > 95% (target > 99% with optimized weights)
- CLEAR readout achieving faster cavity reset than passive ring-down
- Optimized readout parameters (amplitude, length, frequency, weights) persisted

## Prerequisites

- **Notebook 05** — qubit frequency and π-pulse calibrated
- **Notebook 13** — dispersive shift measured (informs readout frequency optimization)

## 1. Setup — Session Bootstrap

Open the notebook stage and load the coherence experiments checkpoint.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    CalibrateReadoutFull,
    IQBlob,
    ReadoutButterflyMeasurement,
    ReadoutGEDiscrimination,
    ReadoutWeightsOptimization,
    fit_quality_gate,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="16_readout_calibration",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

coherence_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="06_coherence_experiments",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"Closed a live in-memory session before reopen: {stage.had_live_session}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
print(f"Current readout frequency: {float(attr.ro_fq) / 1e9:.6f} GHz")
if coherence_checkpoint is not None:
    print(
        "Prior stage 06 status: "
        f"{coherence_checkpoint['status']}"
        f" ({coherence_checkpoint['summary']})"
    )

## 2. Configuration — Readout Calibration Defaults

Define averaging counts, calibration-apply flags, and target parameters for each step of the readout pipeline.

In [ ]:
APPLY_DISCRIMINATION_CALIBRATION = True
APPLY_READOUT_WEIGHTS_CALIBRATION = True
APPLY_FULL_READOUT_CALIBRATION = True

# --- IQ Blob defaults (Exp 14) ---
IQ_BLOB_N_AVG = 10000

# --- Discrimination defaults (Exp 43) ---
DISC_N_AVG = 50000

# --- Raw trace defaults (Exp 42) ---
RAW_TRACE_N_AVG = 10000

# --- Butterfly defaults (Exp 47) ---
BUTTERFLY_N_AVG = 50000

# --- Weight optimization defaults (Exp 53) ---
WEIGHTS_OPT_N_AVG = 100000

# --- Full calibration defaults (Exp 54) ---
FULL_CAL_N_AVG = 100000

print("Readout calibration settings:")
print(f"  apply discrimination calibration: {APPLY_DISCRIMINATION_CALIBRATION}")
print(f"  apply readout weights calibration: {APPLY_READOUT_WEIGHTS_CALIBRATION}")
print(f"  apply full readout calibration: {APPLY_FULL_READOUT_CALIBRATION}")
print(f"  IQ blob n_avg: {IQ_BLOB_N_AVG}")
print(f"  discrimination n_avg: {DISC_N_AVG}")
print(f"  butterfly n_avg: {BUTTERFLY_N_AVG}")

## 3. Execution — IQ Blob (g/e/f State Scatter)

Prepare the qubit in |g⟩, |e⟩, and |f⟩ states and plot the resulting IQ distributions. This provides the initial picture of state separation in the readout plane.

In [ ]:
iq_blob = IQBlob(session)
iq_blob_result = iq_blob.run(n_avg=IQ_BLOB_N_AVG)
iq_blob_analysis = iq_blob.analyze(iq_blob_result, update_calibration=False)
iq_blob.plot(iq_blob_analysis)

print("IQ Blob characterization complete.")
if hasattr(iq_blob_analysis, "metrics"):
    for k, v in iq_blob_analysis.metrics.items():
        print(f"  {k}: {v}")

## 4. Execution — CLEAR Readout Waveform Registration

Register the CLEAR (cavity-level-emission-and-reset) readout waveform for actively depopulating the readout resonator after measurement. *(Pending legacy wrapper confirmation.)*

In [ ]:
# TODO: CLEAR waveform definition depends on legacy readout_mod.
# Build CLEAR pulse and register via pom.create_readout_pulse()
# once the legacy wrapper is confirmed.

print("CLEAR readout registration — TODO: complete once readout_mod API confirmed.")

## 5. Execution — Square Readout Baseline

Measure the integrated readout trace with a standard square readout pulse. This serves as the baseline for CLEAR comparisons.

In [ ]:
# readout_ge_integrated_trace with square readout
ro_sq_result = session.readout_ge_integrated_trace(
    readout_op="readout",
    n_avg=RAW_TRACE_N_AVG,
)

print("Square readout baseline captured.")

## 6. Execution — CLEAR Readout Test

Measure the integrated readout trace using the CLEAR waveform and compare ring-down to the square baseline. *(Pending CLEAR registration.)*

In [ ]:
# TODO: Run readout_ge_integrated_trace with CLEAR readout_op
# once CLEAR is registered.

print("CLEAR readout test — TODO: run after CLEAR registration.")

## 7. Execution — Readout g/e Raw Trace

Time-resolved raw IQ trace for ground and excited state readout. Used to verify signal arrival timing and raw SNR before integration.

In [ ]:
ro_raw_result = session.readout_ge_raw_trace(
    n_avg=RAW_TRACE_N_AVG,
)

print("Raw readout traces captured.")

## 8. Execution — Readout g/e Discrimination

Optimize the discrimination threshold between ground and excited states in the IQ plane. The threshold is calibrated and applied if the fidelity metric passes the quality gate.

In [ ]:
disc = ReadoutGEDiscrimination(session)
disc_result = disc.run(n_avg=DISC_N_AVG)
disc_analysis = disc.analyze(disc_result, update_calibration=True)
disc.plot(disc_analysis)

disc_fit_ok = fit_quality_gate(disc_analysis, r_squared_min=0.80)

if disc_fit_ok:
    disc_patch, disc_patch_preview, disc_apply_result = preview_or_apply_patch_ops(
        session,
        reason="Readout g/e discrimination threshold calibration",
        proposed_patch_ops=disc_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_DISCRIMINATION_CALIBRATION,
    )
    if disc_apply_result is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
        if attr is None:
            raise RuntimeError("Calibration applied, but the refreshed cQED attribute snapshot is unavailable.")
else:
    print("WARNING: Discrimination fit quality gate FAILED — skipping calibration apply.")

print(f"Discrimination fidelity: {disc_analysis.metrics.get('fidelity', 'N/A')}")

## 9. Execution — Readout g/e Integrated Trace

Time-resolved integrated IQ readout for verifying the demodulation window and optimal integration length.

In [ ]:
ro_int_result = session.readout_ge_integrated_trace(
    readout_op="readout",
    n_avg=RAW_TRACE_N_AVG,
)

print("Integrated readout trace captured.")

## 10. Execution — Readout κ Measurement

Extract the readout resonator linewidth $\kappa$ from the ring-down envelope of the raw readout trace. *(Analysis pipeline pending.)*

In [ ]:
# Readout kappa is extracted from the raw trace ring-down envelope.
# The fit uses the exponential decay portion of the readout signal.

# TODO: Extract kappa from ro_raw_result ring-down fit once the
# analysis pipeline is available.

print("Readout κ measurement — TODO: complete ring-down fit analysis.")

## 11. Execution — Residual Photon Ramsey

Measure the residual photon population in the readout resonator after a measurement cycle. Excess photons cause qubit dephasing and must be below threshold for high-fidelity experiments.

In [ ]:
rp_result = session.residual_photon_ramsey(
    n_avg=DISC_N_AVG,
)

print("Residual photon Ramsey complete.")

## 12. Execution — Butterfly Measurement

Quantify measurement back-action via the butterfly protocol: prepare → measure → prepare again → measure again. The correlation between the two outcomes characterizes QND-ness of the readout.

In [ ]:
butterfly = ReadoutButterflyMeasurement(session)
butterfly_result = butterfly.run(n_avg=BUTTERFLY_N_AVG)
butterfly_analysis = butterfly.analyze(butterfly_result, update_calibration=False)
butterfly.plot(butterfly_analysis)

print(f"Butterfly measurement complete.")
if hasattr(butterfly_analysis, "metrics"):
    for k, v in butterfly_analysis.metrics.items():
        print(f"  {k}: {v}")

## 13. Execution — Readout Amplitude & Length Optimization

2D sweep of readout pulse amplitude and integration length to maximize the signal-to-noise ratio for state discrimination.

In [ ]:
ro_opt_result = session.readout_amp_len_opt(
    n_avg=DISC_N_AVG,
)

print("Readout amplitude & length optimization complete.")

## 14. Execution — Readout Frequency Optimization

Sweep the readout drive frequency to find the optimal point for g/e discrimination, accounting for the dispersive shift measured in notebook 13.

In [ ]:
ro_freq_opt_result = session.readout_frequency_optimization(
    n_avg=DISC_N_AVG,
)

print("Readout frequency optimization complete.")

## 15. Execution — CLEAR Variant Comparison

Side-by-side comparison of CLEAR vs. square readout using integrated traces and butterfly measurements. *(Pending CLEAR registration.)*

In [ ]:
# TODO: Compare CLEAR vs. square readout side-by-side.
# Requires CLEAR registration from cell 4.

print("CLEAR variant comparison — TODO: run after CLEAR registration.")

## 16. Execution — Readout Weight Optimization

Optimize the integration weight functions (matched filters) to maximize state discrimination fidelity. Calibrated weights are applied if they pass the quality gate.

In [ ]:
weights_opt = ReadoutWeightsOptimization(session)
weights_opt_result = weights_opt.run(n_avg=WEIGHTS_OPT_N_AVG)
weights_opt_analysis = weights_opt.analyze(weights_opt_result, update_calibration=True)
weights_opt.plot(weights_opt_analysis)

weights_fit_ok = fit_quality_gate(weights_opt_analysis, r_squared_min=0.80)

if weights_fit_ok:
    weights_patch, weights_patch_preview, weights_apply_result = preview_or_apply_patch_ops(
        session,
        reason="Readout integration weight optimization",
        proposed_patch_ops=weights_opt_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_READOUT_WEIGHTS_CALIBRATION,
    )
    if weights_apply_result is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
        if attr is None:
            raise RuntimeError("Calibration applied, but the refreshed cQED attribute snapshot is unavailable.")
else:
    print("WARNING: Readout weights fit quality gate FAILED — skipping calibration apply.")

print("Readout weight optimization complete.")

## 17. Execution — Full Readout Calibration

End-to-end readout calibration combining discrimination threshold, integration weights, and optimal readout parameters into a single unified calibration pass.

In [ ]:
full_cal = CalibrateReadoutFull(session)
full_cal_result = full_cal.run(n_avg=FULL_CAL_N_AVG)
full_cal_analysis = full_cal.analyze(full_cal_result, update_calibration=True)
full_cal.plot(full_cal_analysis)

full_cal_fit_ok = fit_quality_gate(full_cal_analysis, r_squared_min=0.80)

if full_cal_fit_ok:
    full_cal_patch, full_cal_patch_preview, full_cal_apply_result = preview_or_apply_patch_ops(
        session,
        reason="Full readout calibration",
        proposed_patch_ops=full_cal_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_FULL_READOUT_CALIBRATION,
    )
    if full_cal_apply_result is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
        if attr is None:
            raise RuntimeError("Calibration applied, but the refreshed cQED attribute snapshot is unavailable.")
else:
    print("WARNING: Full readout calibration fit quality gate FAILED — skipping calibration apply.")

print("Full readout calibration complete.")
if hasattr(full_cal_analysis, "metrics"):
    for k, v in full_cal_analysis.metrics.items():
        print(f"  {k}: {v}")

## 18. Checkpoint — Save Readout Calibration Results

Persist the calibrated readout parameters. This is the gate-keeper checkpoint: all downstream experiments depend on high-fidelity readout.

In [ ]:
disc_applied = "disc_apply_result" in dir() and disc_apply_result is not None
weights_applied = "weights_apply_result" in dir() and weights_apply_result is not None
full_cal_applied = "full_cal_apply_result" in dir() and full_cal_apply_result is not None

status = "calibrated" if (disc_applied and full_cal_applied) else "characterized"

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="16_readout_calibration",
    status=status,
    summary="Complete readout calibration pipeline: IQ blob, discrimination, weights, and full calibration.",
    consumed_inputs={
        "iq_blob_n_avg": IQ_BLOB_N_AVG,
        "disc_n_avg": DISC_N_AVG,
        "butterfly_n_avg": BUTTERFLY_N_AVG,
        "weights_opt_n_avg": WEIGHTS_OPT_N_AVG,
        "full_cal_n_avg": FULL_CAL_N_AVG,
    },
    persisted_outputs={
        "disc_applied": disc_applied,
        "weights_applied": weights_applied,
        "full_cal_applied": full_cal_applied,
    },
    advisory_outputs={
        "disc_fidelity": disc_analysis.metrics.get("fidelity", None) if "disc_analysis" in dir() else None,
    },
    next_stage="17_readout_bayesian_optimization",
    notes=[
        "CLEAR readout registration (Exp 39) and CLEAR comparison (Exp 51) are pending legacy wrapper confirmation.",
        "κ measurement (Exp 45) ring-down fit needs analysis pipeline.",
        "This is the gate-keeper for all downstream experiments.",
    ],
    metrics=dict(full_cal_analysis.metrics) if "full_cal_analysis" in dir() and hasattr(full_cal_analysis, "metrics") else {},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")